# Phase 2 — Worst-case scaling: `C ~ ρ(Ly,D,β)^{Lx}`

**Hypothesis (plan §2, CLAUDE.md §2a).** Because the proposal factorises over the
`Lx` rows, the worst-case weight multiplies across rows:
$$ C \sim \rho(L_y,D,\beta)^{L_x}, \qquad \log C \sim L_x\,\log\rho. $$
`ρ` is the **per-layer** marginal ratio (the quantity the old notebook mislabels
"C"); at criticality `log ρ` grows ~linearly in `Ly` (entanglement deficit
`S_exact ∝ (c/3)log Ly` vs capacity `log D`). We (2.1) measure `ρ` cleanly, (2.2)
measure the true `C` by enumeration and extract the slope `ρ_eff`, (2.3) reconcile
the two, (2.4) relate `ρ` to the discarded Schmidt weight, (2.5) read off `D*(Ly)`.

> The clean, **correctly-named** `compute_rho_plateau`/`rho_tracker` live in
> `tools/tnmh_tools.jl`; `simulation.ipynb` is left as the historical record.


In [ ]:
using Plots, JLD2, Statistics, LinearAlgebra, DelimitedFiles, Printf
include("main.jl"); include("tools/tnmh_tools.jl")
mkpath("results")
set_seed(20240618)
betac = log(1 + sqrt(2)) / 2          # β_c = ln(1+√2)/2 ≈ 0.4407
log_finding(s) = open(io -> println(io, s), "results/FINDINGS.md", "a")
println("ready. β_c = ", round(betac, digits=6))


## 2.1 — `ρ(Ly, D, β)` surfaces

`compute_rho_plateau(Lx, Ly, β, D)` saturates the boundary environment (length
`Lx`) and returns the per-layer ratio `ρ`. Expect `log ρ` ~ linear in `Ly` at
`β_c` and `ρ → 1` as `D → 2^{⌈Ly/2⌉}`. *(Extend `Ly_array` for publication-quality
curves; large `Ly` is the wall — exact env needs bond `2^{Ly}`.)*


In [ ]:
Lx_sat   = 40
Ly_array = [4, 6, 8, 10]            # extend to 12,14,16 if patient
Ds       = [2, 3, 4]
rho = Dict(D => Float64[] for D in Ds)
for Ly in Ly_array, D in Ds
    push!(rho[D], compute_rho_plateau(Lx_sat, Ly, betac, D))
end
plt = plot(title="ρ(Ly, D, β_c) — per-layer rate", xlabel="L_y (strip width)",
           ylabel="ρ", yscale=:log10, legend=:topleft)
for D in Ds
    plot!(plt, Ly_array, rho[D], marker=:circle, lw=2, label="D = $D")
end
savefig(plt, "results/phase2_rho_vs_Ly.png"); plt


In [ ]:
betas = betac .* [0.5, 0.8, 0.9, 1.0, 1.1, 1.3]
rho_beta = [compute_rho_plateau(40, 8, b, 2) for b in betas]
plt = plot(betas, rho_beta, marker=:circle, lw=2, legend=false,
           xlabel="β", ylabel="ρ(L_y=8, D=2)", title="ρ peaks near β_c")
vline!(plt, [betac], ls=:dash); savefig(plt, "results/phase2_rho_vs_beta.png"); plt


## 2.2 — Brute-force `C(Lx)` and the slope `ρ_eff`

At fixed small `Ly` (enumeration must fit), sweep `Lx` and compute the true
`C`. `log C` vs `Lx` should be a straight line of slope `log ρ_eff` — the direct
confirmation of `C ~ ρ^{Lx}`. *(N = Lx·Ly ≤ 16 here; `Lx=5` ⇒ N=20 ≈ 10⁶ configs,
feasible but slow — uncomment if patient.)*


In [ ]:
Ly = 4; D = 2
Lx_list = [2, 3, 4]                 # push!(Lx_list, 5) for N=20 (slow)
logC = Float64[]
for Lx in Lx_list
    push!(logC, log(enumerate_weights(Lx, Ly, betac, D).C))
end
Amat = hcat(ones(length(Lx_list)), Float64.(Lx_list))
coef = Amat \ logC
slope = coef[2]; rho_eff = exp(slope)
@printf("slope d(log C)/dLx = %.6f   ⇒   ρ_eff = %.6f\n", slope, rho_eff)
plt = scatter(Lx_list, logC, label="log C (brute force)", legend=:topleft)
plot!(plt, Lx_list, Amat * coef, lw=2, label="fit (slope=$(round(slope, digits=4)))")
xlabel!(plt, "L_x"); ylabel!(plt, "log C"); title!(plt, "log C linear in L_x ⇒ C ~ ρ^{L_x}")
savefig(plt, "results/phase2_logC_vs_Lx.png"); plt


## 2.3 — Reconcile `ρ_eff` (conditional, from the slope) vs `ρ_marginal` (the tracker)

The plateau tracker measures a **marginal** single-row ratio (upper bond
`[1,1]`-capped, CLAUDE.md §9.2); the slope of `log C` measures the **conditional**
per-row factor that actually enters `w`. If they agree, the marginal cap is
harmless; if `ρ_eff > ρ_marginal`, the conditional is the right object.


In [ ]:
rho_marg = compute_rho_plateau(40, Ly, betac, D)
@printf("ρ_marginal (plateau, Ly=%d, D=%d) = %.6f\n", Ly, D, rho_marg)
@printf("ρ_eff      (slope of log C)      = %.6f\n", rho_eff)
@printf("ratio ρ_eff / ρ_marginal         = %.4f\n", rho_eff / rho_marg)
println(rho_eff > 1.05 * rho_marg ? "→ conditional exceeds marginal: switch the tracker to condition on rows above." :
                                    "→ marginal ≈ conditional at this (Ly,D): the [1,1] cap is harmless here.")


## 2.4 — `ρ` vs the discarded Schmidt weight `ε_D`

`truncation_error_fidelity` returns `ε_D = sqrt(1 − |⟨e|e_D⟩|²)` between the exact
and `D`-truncated saturated boundary MPS. Relating `ρ−1` to `ε_D` ties the mixing
constant to a measurable truncation diagnostic.


In [ ]:
Lx = 40; Ly = 8
Ds2 = [2, 3, 4, 6]
epsD = [truncation_error_fidelity(Lx, Ly, betac, D) for D in Ds2]
rhoD = [compute_rho_plateau(Lx, Ly, betac, D) for D in Ds2]
@printf("ε_D : %s\nρ-1 : %s\n", string(round.(epsD, sigdigits=3)), string(round.(rhoD .- 1, sigdigits=3)))
plt = scatter(epsD, rhoD .- 1, label="data", xlabel="ε_D (discarded weight)",
              ylabel="ρ − 1", title="per-layer rate vs truncation error (Ly=8, β_c)")
savefig(plt, "results/phase2_rho_vs_epsD.png"); plt


## 2.5 — Bond dimension verdict `D*(Ly)`

Two thresholds: `D*` such that `ρ ≤ 1 + 1/Lx` (per-layer rate controlled ⇒
`C = O(1)`), and the `D` with `ρ = 1` (size-independent `C`, needs `D ~ 2^{Ly/2}`).


In [ ]:
Lx = 40
println("Ly   D*(ρ≤1+1/Lx)   D(ρ=1)")
Dstar_list = Int[]; Dexact_list = Int[]
for Ly in [4, 6, 8]
    Dstar = -1; Dexact = -1
    for D in 2:(2^Ly)
        r = compute_rho_plateau(Lx, Ly, betac, D)
        (Dstar < 0 && r <= 1 + 1/Lx) && (Dstar = D)
        if r <= 1 + 1e-6
            Dexact = D; break
        end
    end
    push!(Dstar_list, Dstar); push!(Dexact_list, Dexact)
    @printf("%-4d %-14d %d\n", Ly, Dstar, Dexact)
end


## Save + record finding

In [ ]:
jldsave("results/phase2_scaling.jld2";
        Ly_array=Ly_array, rho_D2=rho[2], rho_D3=rho[3], rho_D4=rho[4],
        betas=collect(betas), rho_beta=rho_beta,
        Lx_list=Lx_list, logC=logC, slope=slope, rho_eff=rho_eff, rho_marg=rho_marg,
        epsD=epsD, rhoD=rhoD, Dstar=Dstar_list, Dexact=Dexact_list)
log_finding("\n## Phase 2 — Worst-case scaling")
log_finding("- log C linear in Lx: slope=$(round(slope,digits=4)) ⇒ ρ_eff=$(round(rho_eff,digits=6)); ρ_marginal=$(round(rho_marg,digits=6)) (ratio $(round(rho_eff/rho_marg,digits=3))).")
log_finding("- ρ rises with Ly (exponential in width at β_c) and falls with D; D*(Ly) and D(ρ=1) tabulated. See results/phase2_*.png.")
println("saved results/phase2_scaling.jld2")
